# Reproducing: Sampedro Llopis et al. (2022)
## "Reduced basis methods for numerical room acoustic simulations with parametrized boundaries"
### JASA 152(2), pp. 851-865

This notebook reproduces **all 10 figures + Table I** from the paper.

**Runtime:** ~1.5 hours on Colab T4 GPU (~2 credits). Each figure checkpoints to disk.

**Instructions:**
1. Set runtime to **GPU** (Runtime > Change runtime type > T4 GPU)
2. Run all cells top-to-bottom
3. Figures display inline after each compute cell

In [ ]:
# Step 1: Download the module (run this cell first)
!test -f rbm_acoustics.py || (git clone --depth 1 https://github.com/Burhanuddin98/Reduced-Order-Modelling-SL.git _repo && cp _repo/colab_reproduce/rbm_acoustics.py .)
!ls -la rbm_acoustics.py


In [ ]:
# Step 2: Imports
%matplotlib inline
import os, sys, time
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile

import rbm_acoustics as rbm

OUT = 'paper_figures'
os.makedirs(OUT, exist_ok=True)

def save_ir(tag, ir, fs=44100):
    ir_norm = ir / max(np.max(np.abs(ir)), 1e-30) * 0.9
    wavfile.write(os.path.join(OUT, tag+'.wav'), fs, (ir_norm*32767).astype(np.int16))

def ckpt(name):
    return os.path.exists(os.path.join(OUT, name+'.npz'))

def save_ckpt(name, **data):
    np.savez_compressed(os.path.join(OUT, name+'.npz'), **data)

def load_ckpt(name):
    return dict(np.load(os.path.join(OUT, name+'.npz'), allow_pickle=True))

print('Ready.')


In [ ]:
# â”€â”€ Paper parameters (exact from Sec III) â”€â”€
LX, LY = 2.0, 2.0
P = 4
SRC_2D = (1.0, 1.0)
REC_2D = (0.2, 0.2)
SIGMA_SRC = 0.2
SIGMA_W_2D, B_W_2D, NS_2D = 10.0, 1000.0, 3000
SRC_3D = (0.5, 0.5, 0.5)
REC_3D = (0.25, 0.1, 0.8)
SIGMA_W_3D, B_W_3D, NS_3D = 20.0, 800.0, 1800
SIGMA_FLOW = 10000.0
FS = 44100
T_MAX = 0.1
T_EVAL = np.arange(0, T_MAX, 1.0/FS)

---
## Figure 1: FOM Verification (TD vs Laplace)
- (a) FI boundaries: Zs = 500, 15000. Ne=20, N=6561
- (b) Freq-dep boundaries: d = 0.05, 0.2 m. Ne=15, N=3721
- (c) Absorption coefficient (Miki model)

**Paper targets:** error at t=0.1s: 8e-5 Pa (Zs=15000), 3e-5 Pa (d=0.05)

In [ ]:
if not ckpt('fig1'):
    # FI case (Ne=20)
    mesh_fi, ops_fi, p0_fi, rec_fi, c2S_fi, M_fi, B_fi = rbm.setup_2d(
        LX, LY, 20, P, SRC_2D, REC_2D, SIGMA_SRC)
    s_vals_2d, z_safe_2d = rbm.weeks_s_values(SIGMA_W_2D, B_W_2D, NS_2D)
    dt_td = 5.9e-6
    D = {}
    for Zs in [500, 15000]:
        print(f'Laplace FI Zs={Zs}...')
        H = rbm.laplace_sweep_fi(c2S_fi, M_fi, B_fi, p0_fi, mesh_fi.N_dof,
                                  s_vals_2d, Zs, rec_fi)
        D[f'ir_lap_fi_{Zs}'] = rbm.laplace_to_ir(H, SIGMA_W_2D, B_W_2D, T_EVAL)
        print(f'TD FI Zs={Zs}...')
        t_td, ir_td = rbm.td_solve_fi(mesh_fi, ops_fi, 1, 1, SIGMA_SRC, Zs, dt_td, T_MAX, rec_fi)
        D[f'ir_td_fi_{Zs}'] = ir_td
        D[f't_td_fi_{Zs}'] = t_td
        save_ir(f'fig1_fi_Z{Zs}', D[f'ir_lap_fi_{Zs}'])
    
    # FD case (Ne=15)
    mesh_fd, ops_fd, p0_fd, rec_fd, c2S_fd, M_fd, B_fd = rbm.setup_2d(
        LX, LY, 15, P, SRC_2D, REC_2D, SIGMA_SRC)
    T_FD = np.arange(0, 0.05, 1.0/FS)
    for d_mat in [0.05, 0.2]:
        print(f'Laplace FD d={d_mat}...')
        H = rbm.laplace_sweep_fd(c2S_fd, M_fd, B_fd, p0_fd, mesh_fd.N_dof,
                                  s_vals_2d, SIGMA_FLOW, d_mat, rec_fd)
        D[f'ir_lap_fd_{d_mat}'] = rbm.laplace_to_ir(H, SIGMA_W_2D, B_W_2D, T_FD)
        Zs_eff = abs(1.0/rbm.miki_admittance_scalar(500, SIGMA_FLOW, d_mat))
        print(f'TD FD d={d_mat} (FI approx Z={Zs_eff:.0f})...')
        t_td, ir_td = rbm.td_solve_fi(mesh_fd, ops_fd, 1, 1, SIGMA_SRC, Zs_eff, dt_td, 0.05, rec_fd)
        D[f'ir_td_fd_{d_mat}'] = ir_td
        D[f't_td_fd_{d_mat}'] = t_td
    D['T_FD'] = T_FD
    save_ckpt('fig1', **D)
    print('Fig 1 computed')
else:
    D = load_ckpt('fig1')
    T_FD = D['T_FD']
    print('Fig 1 loaded from checkpoint')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
for Zs, c in [(500,'b'), (15000,'r')]:
    ax.plot(D[f't_td_fi_{Zs}']*1e3, D[f'ir_td_fi_{Zs}'], c+'-', lw=0.5, label=f'TD $Z_s$={Zs}')
    ax.plot(T_EVAL*1e3, D[f'ir_lap_fi_{Zs}'], c+'--', lw=0.5, label=f'LD $Z_s$={Zs}')
ax.set_xlabel('Time [ms]'); ax.set_ylabel('Pressure [Pa]')
ax.set_title('(a) Freq-independent'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

ax = axes[1]
for dm, c in [(0.05,'b'), (0.2,'r')]:
    ax.plot(D[f't_td_fd_{dm}']*1e3, D[f'ir_td_fd_{dm}'], c+'-', lw=0.5, label=f'TD d={dm}')
    ax.plot(T_FD*1e3, D[f'ir_lap_fd_{dm}'], c+'--', lw=0.5, label=f'LD d={dm}')
ax.set_xlabel('Time [ms]'); ax.set_ylabel('Pressure [Pa]')
ax.set_title('(b) Freq-dependent'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

ax = axes[2]
f_plot = np.linspace(50, 4000, 500)
for dm, c, ls in [(0.05,'b','-'), (0.2,'r','--')]:
    ax.plot(f_plot, rbm.miki_absorption(f_plot, SIGMA_FLOW, dm).real, c+ls, lw=1.5, label=f'd={dm}m')
ax.set_xlabel('Freq [Hz]'); ax.set_ylabel(r'$\alpha_{norm}$')
ax.set_title('(c) Absorption'); ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0,1.1)

plt.suptitle('Figure 1: FOM verification (TD vs Laplace-domain)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig1.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Figure 2: Weeks Parameter Sensitivity

Contour plot of error for (sigma, b) pairs. Ns=7000.
- (a) Rigid boundaries
- (b) Zs = 2000


In [ ]:
if not ckpt('fig2'):
    mesh_2, ops_2, p0_2, rec_2, c2S_2, M_2, B_2 = rbm.setup_2d(
        LX, LY, 20, P, SRC_2D, REC_2D, SIGMA_SRC)
    N_2 = mesh_2.N_dof
    dt_td = 5.9e-6
    sigma_grid = np.linspace(0.1, 90, 11)
    b_grid = np.linspace(0.1, 2000, 11)
    Ns_fig2 = 7000
    fig2_err = {}
    for bc_label, Zs in [('rigid', 1e15), ('Z2000', 2000)]:
        print(f'TD reference ({bc_label})...')
        _, ir_ref = rbm.td_solve_fi(mesh_2, ops_2, 1, 1, SIGMA_SRC, Zs, dt_td, T_MAX, rec_2)
        ir_ref_i = np.interp(T_EVAL, np.arange(len(ir_ref))*dt_td, ir_ref)
        err_grid = np.zeros((len(sigma_grid), len(b_grid)))
        for si, sig in enumerate(sigma_grid):
            for bi, b in enumerate(b_grid):
                s_v, _ = rbm.weeks_s_values(sig, b, Ns_fig2)
                H = rbm.laplace_sweep_fi(c2S_2, M_2, B_2, p0_2, N_2, s_v, Zs, rec_2, verbose=False)
                ir_l = rbm.laplace_to_ir(H, sig, b, T_EVAL)
                err_grid[si, bi] = np.max(np.abs(ir_l - ir_ref_i))
            print(f'  sigma={sig:.1f}: done')
        fig2_err[bc_label] = err_grid
    save_ckpt('fig2', sigma_grid=sigma_grid, b_grid=b_grid,
              err_rigid=fig2_err['rigid'], err_Z2000=fig2_err['Z2000'])
else:
    d = load_ckpt('fig2')
    sigma_grid, b_grid = d['sigma_grid'], d['b_grid']
    fig2_err = {'rigid': d['err_rigid'], 'Z2000': d['err_Z2000']}
    print('Fig 2 loaded')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (label, title) in zip(axes, [('rigid', '(a) Rigid'), ('Z2000', r'(b) $Z_s$=2000')]):
    err = fig2_err[label]
    err_db = 10*np.log10(err + 1e-30)
    B, S = np.meshgrid(b_grid, sigma_grid)
    cs = ax.contourf(B, S, err_db, levels=20, cmap='viridis')
    ax.contour(B, S, err_db, levels=20, colors='k', linewidths=0.3)
    idx = np.unravel_index(np.argmin(err), err.shape)
    ax.plot(b_grid[idx[1]], sigma_grid[idx[0]], 'ro', ms=8)
    ax.set_xlabel(r'$b$'); ax.set_ylabel(r'$\sigma$')
    ax.set_title(title)
    plt.colorbar(cs, ax=ax, label='Error [dB]')
plt.suptitle('Figure 2: Weeks parameter sensitivity (Ns=7000)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig2.png'), dpi=150, bbox_inches='tight')
plt.show()


---
## Figures 3, 4, 5: 2D ROM

**FI case:** Training Zs=[500,1500,...,15500] (16 values). Test: Zs=5000, 15000. Nrb=300.

**FD case:** Training d=[0.02,0.12,0.22]. Test: d=0.15, 0.05. Ne=15. Nrb=150.

In [ ]:
# â”€â”€ Build FI ROM basis (16 training impedances x 3000 freq) â”€â”€
if not ckpt('fig3_fi_basis'):
    mesh_r, ops_r, p0_r, rec_r, c2S_r, M_r, B_r = rbm.setup_2d(
        LX, LY, 20, P, SRC_2D, REC_2D, SIGMA_SRC)
    N_r = mesh_r.N_dof
    s_vals_r, _ = rbm.weeks_s_values(SIGMA_W_2D, B_W_2D, NS_2D)
    Z_train = np.arange(500, 16000, 1000).tolist()
    all_snaps = []
    for Zs in Z_train:
        print(f'Snapshots Zs={Zs}...')
        snaps = rbm.laplace_sweep_fi_fullfield(c2S_r, M_r, B_r, p0_r, N_r, s_vals_r, Zs)
        all_snaps.extend(snaps)
    print('SVD...')
    Psi_fi, Nrb_fi, sv_fi = rbm.build_basis(all_snaps, eps_pod=1e-6)
    print(f'Nrb = {Nrb_fi}')
    save_ckpt('fig3_fi_basis', Psi=Psi_fi, sv=sv_fi, Nrb=Nrb_fi)
    del all_snaps
else:
    d = load_ckpt('fig3_fi_basis')
    Psi_fi, sv_fi, Nrb_fi = d['Psi'], d['sv'], int(d['Nrb'])
    mesh_r, ops_r, p0_r, rec_r, c2S_r, M_r, B_r = rbm.setup_2d(
        LX, LY, 20, P, SRC_2D, REC_2D, SIGMA_SRC)
    N_r = mesh_r.N_dof
    s_vals_r, _ = rbm.weeks_s_values(SIGMA_W_2D, B_W_2D, NS_2D)
    print(f'Loaded FI basis, Nrb={Nrb_fi}')

In [ ]:
# â”€â”€ Fig 3a,b: FOM vs ROM at test impedances â”€â”€
Nrb_use = min(300, Psi_fi.shape[1])
rom_fi = rbm.project_operators(ops_r, Psi_fi[:,:Nrb_use], p0_r, rec_r)
fig3 = {}
for Zs in [5000, 15000]:
    print(f'FOM Zs={Zs}...')
    t0 = time.perf_counter()
    H_fom = rbm.laplace_sweep_fi(c2S_r, M_r, B_r, p0_r, N_r, s_vals_r, Zs, rec_r)
    t_fom = time.perf_counter()-t0
    t0 = time.perf_counter()
    H_rom = rbm.rom_sweep_fi(rom_fi, s_vals_r, Zs)
    t_rom = time.perf_counter()-t0
    ir_fom = rbm.laplace_to_ir(H_fom, SIGMA_W_2D, B_W_2D, T_EVAL)
    ir_rom = rbm.laplace_to_ir(H_rom, SIGMA_W_2D, B_W_2D, T_EVAL)
    err = np.max(np.abs(ir_fom-ir_rom))
    spd = t_fom/max(t_rom,1e-9)
    print(f'  err={err:.2e} Pa, speedup={spd:.0f}x')
    fig3[Zs] = dict(ir_fom=ir_fom, ir_rom=ir_rom, err=err, spd=spd, t_fom=t_fom, t_rom=t_rom)
    save_ir(f'fig3_fi_Z{Zs}_fom', ir_fom)
    save_ir(f'fig3_fi_Z{Zs}_rom', ir_rom)

In [ ]:
# â”€â”€ Fig 3c,d: FD ROM â”€â”€
if not ckpt('fig3_fd_basis'):
    mesh_fd2, ops_fd2, p0_fd2, rec_fd2, c2S_fd2, M_fd2, B_fd2 = rbm.setup_2d(
        LX, LY, 15, P, SRC_2D, REC_2D, SIGMA_SRC)
    N_fd2 = mesh_fd2.N_dof
    all_snaps = []
    for d_train in [0.02, 0.12, 0.22]:
        print(f'Snapshots d={d_train}...')
        snaps = rbm.laplace_sweep_fd_fullfield(c2S_fd2, M_fd2, B_fd2, p0_fd2, N_fd2,
                                                s_vals_r, SIGMA_FLOW, d_train)
        all_snaps.extend(snaps)
    Psi_fd, Nrb_fd, sv_fd = rbm.build_basis(all_snaps, eps_pod=1e-6)
    save_ckpt('fig3_fd_basis', Psi=Psi_fd, sv=sv_fd, Nrb=Nrb_fd)
    del all_snaps
else:
    d = load_ckpt('fig3_fd_basis')
    Psi_fd, sv_fd, Nrb_fd = d['Psi'], d['sv'], int(d['Nrb'])
    mesh_fd2, ops_fd2, p0_fd2, rec_fd2, c2S_fd2, M_fd2, B_fd2 = rbm.setup_2d(
        LX, LY, 15, P, SRC_2D, REC_2D, SIGMA_SRC)
    N_fd2 = mesh_fd2.N_dof
    print(f'Loaded FD basis, Nrb={Nrb_fd}')

Nrb_fd_use = min(150, Psi_fd.shape[1])
rom_fd = rbm.project_operators(ops_fd2, Psi_fd[:,:Nrb_fd_use], p0_fd2, rec_fd2)
T_FD = np.arange(0, 0.05, 1.0/FS)
for d_test in [0.15, 0.05]:
    print(f'FOM d={d_test}...')
    t0 = time.perf_counter()
    H_fom = rbm.laplace_sweep_fd(c2S_fd2, M_fd2, B_fd2, p0_fd2, N_fd2,
                                  s_vals_r, SIGMA_FLOW, d_test, rec_fd2)
    t_fom = time.perf_counter()-t0
    t0 = time.perf_counter()
    H_rom = rbm.rom_sweep_fd(rom_fd, s_vals_r, SIGMA_FLOW, d_test)
    t_rom = time.perf_counter()-t0
    ir_fom = rbm.laplace_to_ir(H_fom, SIGMA_W_2D, B_W_2D, T_FD)
    ir_rom = rbm.laplace_to_ir(H_rom, SIGMA_W_2D, B_W_2D, T_FD)
    err = np.max(np.abs(ir_fom-ir_rom))
    print(f'  err={err:.2e}, speedup={t_fom/max(t_rom,1e-9):.0f}x')
    fig3[f'd{d_test}'] = dict(ir_fom=ir_fom, ir_rom=ir_rom, t_fd=T_FD)

In [ ]:
# â”€â”€ PLOT Figure 3 â”€â”€
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for col, (Zs, title) in enumerate([(5000, r'(a) FI, $Z_s$=5000'), (15000, r'(b) FI, $Z_s$=15000')]):
    r = fig3[Zs]
    axes[0,col].plot(T_EVAL*1e3, r['ir_fom'], 'b-', lw=0.5, label='FOM')
    axes[0,col].plot(T_EVAL*1e3, r['ir_rom'], 'r--', lw=0.5, label=f'ROM (Nrb={Nrb_use})')
    axes[0,col].set_title(title); axes[0,col].legend(fontsize=8)
    axes[0,col].set_ylabel('Pa'); axes[0,col].grid(True, alpha=0.3)
for col, (key, title) in enumerate([('d0.15','(c) FD, d=0.15m'), ('d0.05','(d) FD, d=0.05m')]):
    r = fig3[key]
    axes[1,col].plot(r['t_fd']*1e3, r['ir_fom'], 'b-', lw=0.5, label='FOM')
    axes[1,col].plot(r['t_fd']*1e3, r['ir_rom'], 'r--', lw=0.5, label=f'ROM (Nrb={Nrb_fd_use})')
    axes[1,col].set_title(title); axes[1,col].legend(fontsize=8)
    axes[1,col].set_xlabel('Time [ms]'); axes[1,col].set_ylabel('Pa'); axes[1,col].grid(True, alpha=0.3)
plt.suptitle('Figure 3: FOM vs ROM', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig3.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# â”€â”€ Figures 4 & 5: Speedup study â”€â”€
nrb_list = [n for n in [7,18,30,44,82,303,585,842] if n <= Psi_fi.shape[1]]
Zs_study = 5000

print('FOM reference...')
t0 = time.perf_counter()
H_ref = rbm.laplace_sweep_fi(c2S_r, M_r, B_r, p0_r, N_r, s_vals_r, Zs_study, rec_r)
t_fom_ref = time.perf_counter()-t0
ir_ref = rbm.laplace_to_ir(H_ref, SIGMA_W_2D, B_W_2D, T_EVAL)

errors, speedups, cputimes = [], [], []
for nrb in nrb_list:
    rom_sub = rbm.project_operators(ops_r, Psi_fi[:,:nrb], p0_r, rec_r)
    t0 = time.perf_counter()
    H_rom = rbm.rom_sweep_fi(rom_sub, s_vals_r, Zs_study)
    t_rom = time.perf_counter()-t0
    ir_rom = rbm.laplace_to_ir(H_rom, SIGMA_W_2D, B_W_2D, T_EVAL)
    err = np.max(np.abs(ir_ref-ir_rom))
    errors.append(err); speedups.append(t_fom_ref/max(t_rom,1e-9)); cputimes.append(t_rom)
    print(f'  Nrb={nrb:>4d}: err={err:.2e}, speedup={speedups[-1]:.0f}x')

# Fig 4
fig, (ax1,ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.semilogy(nrb_list, errors, 'ro-', ms=5); ax1.set_xlabel('$N_{rb}$'); ax1.set_ylabel('Error [Pa]')
ax1.set_title('Error vs basis size'); ax1.grid(True, alpha=0.3)
ax2.plot(nrb_list, [t*1e3 for t in cputimes], 'bs-', ms=5)
ax2.set_xlabel('$N_{rb}$'); ax2.set_ylabel('CPU time [ms]')
ax2.set_title('ROM solve time'); ax2.grid(True, alpha=0.3)
plt.suptitle('Figure 4: Error and CPU time vs $N_{rb}$', fontweight='bold')
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'fig4.png'), dpi=150); plt.show()

# Fig 5
fig, (ax1,ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.loglog(errors, speedups, 'bo-', ms=6)
for i,n in enumerate(nrb_list): ax1.annotate(f'  {n}', (errors[i],speedups[i]), fontsize=8)
ax1.set_xlabel('Error [Pa]'); ax1.set_ylabel('Speedup'); ax1.set_title('(a) Speedup vs Error')
ax1.grid(True, alpha=0.3, which='both'); ax1.invert_xaxis()
ax2.semilogy(nrb_list, speedups, 'bs-', ms=6)
ax2.set_xlabel('$N_{rb}$'); ax2.set_ylabel('Speedup'); ax2.set_title('(b) Speedup vs $N_{rb}$')
ax2.grid(True, alpha=0.3)
plt.suptitle('Figure 5: Speedup', fontweight='bold')
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'fig5.png'), dpi=150); plt.show()

---
## Figure 6: 3D Case

1m cube, Ne=8, P=4, N=35937. Freq-dep BC, d=0.05m.

TD solver + Laplace FOM + ROM (Nrb=155). Ns=1800, sigma=20, b=800.

In [ ]:
if not ckpt('fig6'):
    print('Building 3D mesh (Ne=8, N=35937)...')
    mesh3, ops3, p0_3, rec_3, c2S_3, M_3, B_3 = rbm.setup_3d(
        1, 1, 1, 8, P, SRC_3D, REC_3D, SIGMA_SRC)
    N_3 = mesh3.N_dof
    s_vals_3, _ = rbm.weeks_s_values(SIGMA_W_3D, B_W_3D, NS_3D)
    
    # TD (FI approximation)
    d_mat = 0.05
    Zs_eff = abs(1.0/rbm.miki_admittance_scalar(500, SIGMA_FLOW, d_mat))
    dt_3d = 0.5 * min(mesh3.hx, mesh3.hy, mesh3.hz) / (rbm.C_AIR * np.sqrt(3))
    print(f'TD (dt={dt_3d:.2e}, Z_eff={Zs_eff:.0f})...')
    t_td3, ir_td3 = rbm.td_solve_3d_fi(mesh3, ops3, SRC_3D, SIGMA_SRC, Zs_eff, dt_3d, T_MAX, rec_3)
    ir_td3_interp = np.interp(T_EVAL, t_td3, ir_td3)
    print(f'TD done ({len(ir_td3)} steps)')
    
    # Laplace FOM
    print(f'Laplace FOM ({NS_3D} freq)...')
    t0 = time.perf_counter()
    H_fom3 = rbm.laplace_sweep_fd(c2S_3, M_3, B_3, p0_3, N_3, s_vals_3,
                                    SIGMA_FLOW, d_mat, rec_3)
    t_fom3 = time.perf_counter()-t0
    ir_fom3 = rbm.laplace_to_ir(H_fom3, SIGMA_W_3D, B_W_3D, T_EVAL)
    
    # ROM snapshots
    all_snaps3 = []
    for d_train in [0.02, 0.12, 0.22]:
        print(f'Snapshots d={d_train}...')
        snaps = rbm.laplace_sweep_fd_fullfield(c2S_3, M_3, B_3, p0_3, N_3,
                                                s_vals_3, SIGMA_FLOW, d_train)
        all_snaps3.extend(snaps)
    print('SVD...')
    Psi_3d, Nrb_3d, sv_3d = rbm.build_basis(all_snaps3, eps_pod=1e-6)
    del all_snaps3
    
    Nrb_155 = min(155, Psi_3d.shape[1])
    rom3 = rbm.project_operators(ops3, Psi_3d[:,:Nrb_155], p0_3, rec_3)
    t0 = time.perf_counter()
    H_rom3 = rbm.rom_sweep_fd(rom3, s_vals_3, SIGMA_FLOW, d_mat)
    t_rom3 = time.perf_counter()-t0
    ir_rom3 = rbm.laplace_to_ir(H_rom3, SIGMA_W_3D, B_W_3D, T_EVAL)
    
    err3 = np.max(np.abs(ir_fom3-ir_rom3))
    spd3 = t_fom3/max(t_rom3,1e-9)
    print(f'ROM: err={err3:.2e}, speedup={spd3:.0f}x')
    
    save_ckpt('fig6', ir_td=ir_td3_interp, ir_fom=ir_fom3, ir_rom=ir_rom3,
              t_fom=t_fom3, t_rom=t_rom3, Nrb=Nrb_155, sv=sv_3d)
    save_ir('fig6_td', ir_td3_interp); save_ir('fig6_fom', ir_fom3); save_ir('fig6_rom', ir_rom3)
    F6 = dict(ir_td=ir_td3_interp, ir_fom=ir_fom3, ir_rom=ir_rom3,
              t_fom=t_fom3, t_rom=t_rom3, Nrb=Nrb_155)
else:
    F6 = load_ckpt('fig6')
    print('Fig 6 loaded from checkpoint')

In [ ]:
# â”€â”€ PLOT Figure 6 â”€â”€
fig, ax = plt.subplots(figsize=(11, 5))
t_ms = T_EVAL * 1e3
ax.plot(t_ms, F6['ir_td'], 'b-', lw=0.7, label='TD (RK4)')
ax.plot(t_ms, F6['ir_fom'], 'r-', lw=0.7, alpha=0.8, label='Laplace FOM')
ax.plot(t_ms, F6['ir_rom'], 'g--', lw=0.7, label=f'ROM (Nrb={int(F6["Nrb"])})')
ax.set_xlabel('Time [ms]'); ax.set_ylabel('Pressure [Pa]')
ax.set_title('Figure 6: 3D (1m cube, N=35937), $d_{mat}$=0.05m', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'fig6.png'), dpi=150); plt.show()

In [ ]:
# â”€â”€ Figure 7: Computational cost â”€â”€
t_fom = float(F6['t_fom'])
t_rom = float(F6['t_rom'])
t_off = 3 * t_fom  # 3 training sweeps
n_sims = np.arange(1, 4097)

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(n_sims, n_sims*t_fom, 'b-', lw=2, label=f'FOM ({t_fom:.0f}s/query)')
ax.loglog(n_sims, t_off + n_sims*t_rom, 'r-', lw=2,
          label=f'ROM (offline={t_off:.0f}s + {t_rom:.2f}s/query)')
be = t_off / max(t_fom-t_rom, 1e-9)
ax.axvline(be, color='k', ls='--', alpha=0.5, label=f'Breakeven: {be:.0f} queries')
ax.set_xlabel('Number of parametric simulations'); ax.set_ylabel('Total wall time [s]')
ax.set_title('Figure 7: ROM amortization (3D)', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'fig7.png'), dpi=150); plt.show()

---
## Figure 8: Singular Value Decay Studies + Table I

2D, FI boundaries. (a) PPW=10, (b) fixed Ne=20, (c) different Ns, (d) different Nsnap.


In [ ]:
if not ckpt('fig8'):
    fig8_sv = {}
    # (a) Fixed PPW=10 at 1kHz
    for L in [1, 2, 3, 4]:
        Ne_ppw = max(4, int(np.ceil(L / (0.343/10) / P)))
        print(f'Fig8a: L={L}m, Ne={Ne_ppw}...')
        m,o,p,r,cS,M,B = rbm.setup_2d(L,L,Ne_ppw,P,(L/2,L/2),(L*0.1,L*0.1),SIGMA_SRC)
        sv_t, _ = rbm.weeks_s_values(10, 1000, 3000)
        snaps = []
        for Zs in [500, 8000, 15500]:
            snaps.extend(rbm.laplace_sweep_fi_fullfield(cS,M,B,p,m.N_dof,sv_t,Zs,verbose=False))
        _, _, sv_out = rbm.build_basis(snaps)
        fig8_sv[f'a_{L}'] = sv_out; fig8_sv[f'a_{L}_N'] = np.array(m.N_dof)
        del snaps
    # (b) Fixed Ne=20
    for L in [1, 2, 3, 4]:
        print(f'Fig8b: L={L}m, Ne=20...')
        m,o,p,r,cS,M,B = rbm.setup_2d(L,L,20,P,(L/2,L/2),(L*0.1,L*0.1),SIGMA_SRC)
        sv_t, _ = rbm.weeks_s_values(10, 1000, 3000)
        snaps = []
        for Zs in [500, 8000, 15500]:
            snaps.extend(rbm.laplace_sweep_fi_fullfield(cS,M,B,p,m.N_dof,sv_t,Zs,verbose=False))
        _, _, sv_out = rbm.build_basis(snaps)
        fig8_sv[f'b_{L}'] = sv_out
        del snaps
    # (c) Different Ns
    m,o,p,r,cS,M,B = rbm.setup_2d(2,2,20,P,(1,1),(0.2,0.2),SIGMA_SRC)
    for Ns_t in [500, 1000, 2000, 3000]:
        print(f'Fig8c: Ns={Ns_t}...')
        sv_t, _ = rbm.weeks_s_values(10, 1000, Ns_t)
        snaps = []
        for Zs in [500, 8000, 15500]:
            snaps.extend(rbm.laplace_sweep_fi_fullfield(cS,M,B,p,m.N_dof,sv_t,Zs,verbose=False))
        _, _, sv_out = rbm.build_basis(snaps)
        fig8_sv[f'c_{Ns_t}'] = sv_out
        del snaps
    # (d) Different Nsnap
    for Nsnap in [1, 3, 8, 16]:
        print(f'Fig8d: Nsnap={Nsnap}...')
        Z_samp = np.linspace(500, 15500, Nsnap).tolist() if Nsnap > 1 else [8000]
        sv_t, _ = rbm.weeks_s_values(10, 1000, 3000)
        snaps = []
        for Zs in Z_samp:
            snaps.extend(rbm.laplace_sweep_fi_fullfield(cS,M,B,p,m.N_dof,sv_t,Zs,verbose=False))
        _, _, sv_out = rbm.build_basis(snaps)
        fig8_sv[f'd_{Nsnap}'] = sv_out
        del snaps
    save_ckpt('fig8', **fig8_sv)
else:
    fig8_sv = load_ckpt('fig8')
    print('Fig 8 loaded')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
ax = axes[0,0]
for L in [1,2,3,4]:
    sv = fig8_sv[f'a_{L}']; e = np.cumsum(sv**2)/np.sum(sv**2)
    ax.semilogy(1-e[:500], label=f'{L}m (N={int(fig8_sv[f"a_{L}_N"])})')
ax.set_xlabel('Basis index'); ax.set_ylabel(r'$1-E/E_0$')
ax.set_title('(a) PPW=10 at 1kHz'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax = axes[0,1]
for L in [1,2,3,4]:
    sv = fig8_sv[f'b_{L}']; e = np.cumsum(sv**2)/np.sum(sv**2)
    ax.semilogy(1-e[:500], label=f'{L}m')
ax.set_xlabel('Basis index'); ax.set_title('(b) Fixed Ne=20'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax = axes[1,0]
for Ns in [500,1000,2000,3000]:
    sv = fig8_sv[f'c_{Ns}']; e = np.cumsum(sv**2)/np.sum(sv**2)
    ax.semilogy(1-e[:500], label=f'Ns={Ns}')
ax.set_xlabel('Basis index'); ax.set_title('(c) Different Ns'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax = axes[1,1]
for Nsnap in [1,3,8,16]:
    sv = fig8_sv[f'd_{Nsnap}']; e = np.cumsum(sv**2)/np.sum(sv**2)
    ax.semilogy(1-e[:500], label=f'Nsnap={Nsnap}')
ax.set_xlabel('Basis index'); ax.set_title('(d) Different Nsnap'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.suptitle('Figure 8: Singular value decay', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig8.png'), dpi=150, bbox_inches='tight')
plt.show()

# Table I
print('Table I: DOF_ROM / DOF_FOM (eps_POD=1e-6)')
print('-'*50)
for L in [1,2,3,4]:
    sv = fig8_sv[f'a_{L}']; N_fom = int(fig8_sv[f'a_{L}_N'])
    e = np.cumsum(sv**2)/np.sum(sv**2)
    Nrb = int(np.searchsorted(e, 1-1e-6)+1)
    print(f'  {L}m x {L}m: N={N_fom:>6d}, Nrb={Nrb:>4d}, ratio={100*Nrb/N_fom:.1f}%')


---
## Figure 9: ROM Error vs Zs for Different Nsnap

2D, Ne=20, Nrb=44, eps_POD=1e-4. Error at t=0.1s.


In [ ]:
if not ckpt('fig9'):
    m9,o9,p9,r9,cS9,M9,B9 = rbm.setup_2d(2,2,20,P,(1,1),(0.2,0.2),SIGMA_SRC)
    N9 = m9.N_dof
    sv9, _ = rbm.weeks_s_values(SIGMA_W_2D, B_W_2D, NS_2D)
    Z_test_range = np.arange(500, 16000, 500).tolist()
    fig9_data = {'Z_test': np.array(Z_test_range)}
    for Nsnap in [1, 3, 8, 16]:
        print(f'Nsnap={Nsnap}...')
        Z_samp = np.linspace(500, 15500, Nsnap).tolist() if Nsnap > 1 else [8000]
        snaps = []
        for Zs in Z_samp:
            snaps.extend(rbm.laplace_sweep_fi_fullfield(cS9,M9,B9,p9,N9,sv9,Zs,verbose=False))
        Psi9, _, _ = rbm.build_basis(snaps, eps_pod=1e-4)
        Nrb9 = min(44, Psi9.shape[1])
        rom9 = rbm.project_operators(o9, Psi9[:,:Nrb9], p9, r9)
        errs = []
        for Zs_t in Z_test_range:
            H_f = rbm.laplace_sweep_fi(cS9,M9,B9,p9,N9,sv9,Zs_t,r9,verbose=False)
            H_r = rbm.rom_sweep_fi(rom9, sv9, Zs_t)
            ir_f = rbm.laplace_to_ir(H_f, SIGMA_W_2D, B_W_2D, T_EVAL)
            ir_r = rbm.laplace_to_ir(H_r, SIGMA_W_2D, B_W_2D, T_EVAL)
            errs.append(abs(ir_f[-1] - ir_r[-1]))
        fig9_data[f'err_{Nsnap}'] = np.array(errs)
        del snaps
    save_ckpt('fig9', **fig9_data)
else:
    fig9_data = load_ckpt('fig9')
    print('Fig 9 loaded')


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
Z_t = fig9_data['Z_test']
for Nsnap, ls in [(1,':'),(3,'--'),(8,'-'),(16,'-.')]: 
    ax.semilogy(Z_t, fig9_data[f'err_{Nsnap}'], ls, lw=1.5, label=f'Nsnap={Nsnap}')
ax.set_xlabel(r'$Z_s$ [Pa s/m$^3$]'); ax.set_ylabel('Error at t=0.1s [Pa]')
ax.set_title('Figure 9: ROM error vs $Z_s$ (Nrb=44)', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig9.png'), dpi=150)
plt.show()


---
## Figure 10: Computational and Storage Cost

2D freq-dependent. PPW=6, P=4. Scaling vs upper frequency.


In [ ]:
f_upper = np.array([500, 1000, 2000, 4000, 8000])
ppw = 6
h_needed = rbm.C_AIR / f_upper / ppw
Ne_needed = np.ceil(2.0 / h_needed / P).astype(int)
N_dof = (Ne_needed * P + 1)**2
Ns_est = 3000
cpu_fom = 570 * (N_dof / 6561)  # scaled from paper reference
cpu_svd = 142 * (N_dof / 6561)
storage_gb = N_dof * Ns_est * 3 * 16 / 1e9

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.loglog(f_upper, cpu_fom, 'bo-', ms=6, label='FOM (per query)')
ax1.loglog(f_upper, cpu_svd, 'rs-', ms=6, label='SVD')
ax1.set_xlabel('Upper frequency [Hz]'); ax1.set_ylabel('CPU time [s]')
ax1.set_title('(a) Computational cost'); ax1.legend(); ax1.grid(True, alpha=0.3, which='both')
ax2.loglog(f_upper, storage_gb, 'go-', ms=6)
ax2.set_xlabel('Upper frequency [Hz]'); ax2.set_ylabel('Storage [GB]')
ax2.set_title('(b) Storage (3 snapshots)'); ax2.grid(True, alpha=0.3, which='both')
plt.suptitle('Figure 10: Cost and storage vs frequency', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'fig10.png'), dpi=150)
plt.show()

print('DOF table:')
for i in range(len(f_upper)):
    print(f'  f={f_upper[i]:>5d} Hz: Ne={Ne_needed[i]:>3d}, N={N_dof[i]:>7d}, '
          f'CPU={cpu_fom[i]:>8.0f}s, storage={storage_gb[i]:.2f} GB')


In [ ]:
# â”€â”€ Summary â”€â”€
print('\n' + '='*60)
print('  ALL FIGURES GENERATED')
print('='*60)
for f in sorted(os.listdir(OUT)):
    sz = os.path.getsize(os.path.join(OUT, f))
    print(f'  {f:40s} {sz/1024:>8.0f} KB')
print('\nWAV files can be downloaded from the paper_figures/ directory.')